# Frozen-history audit
Harness prepared before the Sprint; GPU runs are Sprint work (see PREEXISTING.md). Select an A100 runtime. Read the CU/hour rate first; allow up to 3 hours (budget 30 CU at 10 CU/hour). Stop if the unchanged-prompt reconstruction fails broadly. Upload the supplied audit-source.zip; no Git push is needed. Model access requires your own authorized Hugging Face account. Keep this browser open and download results before disconnecting.

In [ ]:
from google.colab import files
import zipfile, os
uploaded = files.upload()
assert "audit-source.zip" in uploaded
with zipfile.ZipFile("audit-source.zip") as z:
    z.extractall("/content/frozen-audit")
os.chdir("/content/frozen-audit")


In [ ]:
!python scripts/verify_provenance.py


In [ ]:
# Same transformers as the completed 2x2 run; torch is Colab's (recorded in the identity).
%pip -q install transformers==5.22.1 accelerate numpy


In [ ]:
from huggingface_hub import login
login()  # user completes login; do not put a token in saved cells


In [ ]:
import torch, json, time
assert torch.cuda.is_available(), "Select a GPU runtime"
print(torch.cuda.get_device_name(0))
print("VRAM GB:", torch.cuda.get_device_properties(0).total_memory / 2**30)
m=json.load(open("provenance/checkpoints.json"))
assert len(m["checkpoints"]) == 30
print("Checkpoints:", len(m["checkpoints"]))


In [ ]:
# Pinned to the revision of the completed 2x2 run (2026-11-13) so both runs share weights/template.
MODEL = "meta-llama/Llama-3.1-8B-Instruct"
REVISION = "0e9e39f249a16976918f6564b8830bc894c89659"
print("Pinned revision:", REVISION)


## Neutral-sentence set (Sprint run)
Eight neutral prefixes plus none/discount, fixed in `docs/neutral_set_plan.md`. Stop if the reconstruction gate fails broadly.


In [ ]:
!python scripts/frozen_audit.py prefix-run --set neutral --manifest provenance/checkpoints.json --model {MODEL} --revision {REVISION} --out results/neutral --device cuda


In [ ]:
!python scripts/frozen_audit.py prefix-report --results results/neutral --out results/neutral.csv


## Prefix control: none / discount / placebo / reverse (completed; skip unless re-running)
Sentences and decision rule are fixed in `docs/prefix_control_plan.md`. Stop if the reconstruction gate fails broadly.


In [ ]:
!python scripts/frozen_audit.py prefix-run --manifest provenance/checkpoints.json --model {MODEL} --revision {REVISION} --out results/prefix --device cuda


In [ ]:
!python scripts/frozen_audit.py prefix-report --results results/prefix --out results/prefix.csv


## Inherited 2x2 (already completed; skip unless re-running)


In [ ]:
# Shell magic streams per-checkpoint PASS/failure lines into this cell.
!python scripts/frozen_audit.py run --manifest provenance/checkpoints.json --model {MODEL} --revision {REVISION} --out results/presprint --device cuda


In [ ]:
!python scripts/frozen_audit.py report --results results/presprint --out results/paired.csv


In [ ]:
import shutil
from google.colab import files
shutil.make_archive("/content/neutral-results", "zip", "/content/frozen-audit/results")
files.download("/content/neutral-results.zip")  # if blocked: Files pane, right-click, Download


After download, disconnect and delete the runtime to stop GPU consumption. Return the results ZIP for manual text review and the paired diagnostic analysis. Failed reconstructions remain in the output; do not label them successful audits.